# 01 · TimeSeries ETT —— 看天看表，猜明天多少度

**家族位置**：09 领域应用第 1 站。主干家族（04 LSTM/05 Attention）学完后的综合出口：同一批电力数据，三种思路同台猜未来 24 小时油温。

**学习目标**：滑窗预测口径；LSTM 递推 vs Informer 稀疏 vs PatchTST 切块三路线；MSE 同台对照；稀疏/蒸馏消融。

## 1. 原理：三种看表姿势

### 通俗理解

**一句话**：LSTM 像逐字默读（一页一页翻，看到 96 页猜后 24 页）；Informer 像跳读（只看关键词 top-u，翻页还撕一半）；PatchTST 像剪报（剪成 16 格一块看，每种电表分开看再平均）。

### 结构账

```
数据： ETTh1 小时级 17420×7（6 负载+油温 OT），只猜 OT 未来 24 点；滑窗 96→24
切分： 7:1:2 时序切（行 12194/1742/3484 → 窗 12075/1622/3366），只用训练段定标准化
模型： LSTM(64×2) / InformerMini(稀疏top-u+Conv蒸馏) / PatchTST(P=16,S=8,通道独立)
口径： MSE(标准化域) + 反标准化 OT(℃) RMSE 双报；Adam 1e-3；LSTM/Informer 15ep，PatchTST 8ep（单ep慢8×）
```

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
ROOT=Path.cwd()
while ROOT != ROOT.parent and not (ROOT/'common').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from common.data import load_ett
from common.models import LSTMForecaster, InformerMini, PatchTSTMini
from common.engine import fit_reg, test_mse
from common.utils import set_seed,setup_chinese_font,count_params
set_seed(0); setup_chinese_font()
FIGS=Path.cwd()/'figs'; FIGS.mkdir(exist_ok=True)
print('torch:',torch.__version__)
tr,va,te,cols=load_ett('data/ETTh1.csv',seq_in=96,seq_out=24)
print(f'cols={cols}')
print(f'train {tuple(tr[0].shape)} / val {tuple(va[0].shape)} / test {tuple(te[0].shape)}')
trl=DataLoader(TensorDataset(*tr),batch_size=128,shuffle=True)
val=DataLoader(TensorDataset(*va),batch_size=512)
tel=DataLoader(TensorDataset(*te),batch_size=512)
fig,ax=plt.subplots(figsize=(7,2.6))
ax.plot(np.array(te[1][:3]))
ax.set_title('OT 未来24点样本（标准化域）：周期+噪声，猜趋势不猜毛刺')
ax.set_xlabel('horizon'); ax.set_ylabel('OT (norm)')
plt.tight_layout(); plt.savefig(FIGS/'fig1_data.png',dpi=150,bbox_inches='tight'); plt.show()

## 2. 同台训练：LSTM vs Informer(15ep) vs PatchTST(8ep)

In [ ]:
torch.manual_seed(0); lstm=LSTMForecaster()
torch.manual_seed(0); inf=InformerMini(sparse=True)
torch.manual_seed(0); pts=PatchTSTMini()
print(f'params LSTM={count_params(lstm)} Informer={count_params(inf)} PatchTST={count_params(pts)}',flush=True)
h_lstm=fit_reg(lstm,trl,val,epochs=15,lr=1e-3)
h_inf=fit_reg(inf,trl,val,epochs=15,lr=1e-3)
h_pts=fit_reg(pts,trl,val,epochs=8,lr=1e-3)
fig,ax=plt.subplots(figsize=(6,3.2))
for h,c,l in [(h_lstm,'#4C72B0','LSTM-15ep'),(h_inf,'#55A868','Informer-15ep'),(h_pts,'#DD8452','PatchTST-8ep')]:
    ax.plot([v for _,v in h],label=l,color=c)
ax.set_xlabel('epoch'); ax.set_ylabel('val MSE'); ax.legend()
ax.set_title('同台训练：谁先看懂电力周期')
plt.tight_layout(); plt.savefig(FIGS/'fig2_train.png',dpi=150,bbox_inches='tight'); plt.show()

## 3. 测试对照：MSE + OT(℃) RMSE + 消融

In [ ]:
import pandas as pd
res={}
for name,m in [('LSTM',lstm),('Informer',inf),('PatchTST',pts)]:
    res[name]=test_mse(m,tel)
    print(f'{name} test-MSE={res[name]:.4f}',flush=True)
raw=pd.read_csv('data/ETTh1.csv')['OT'].values[:int(17420*0.7)]
mu,sd=raw.mean(),raw.std()
for name in res: print(f'{name} OT-RMSE={np.sqrt(res[name])*sd:.3f}℃',flush=True)
fig,ax=plt.subplots(figsize=(6,3.2))
ax.bar(list(res),list(res.values()),color=['#4C72B0','#55A868','#DD8452'])
for i,v in enumerate(res.values()): ax.text(i,v+0.005,f'{v:.4f}',ha='center',fontsize=10)
ax.set_ylabel('test MSE'); ax.set_title('同台对照：三路线谁猜得准')
plt.tight_layout(); plt.savefig(FIGS/'fig3_compare.png',dpi=150,bbox_inches='tight'); plt.show()
with torch.no_grad():
    xb=te[0][:1]
    preds={n:m(xb).flatten().numpy()*sd+mu for n,m in [('LSTM',lstm),('Informer',inf),('PatchTST',pts)]}
    truth=te[1][0].numpy()*sd+mu
fig,ax=plt.subplots(figsize=(7,3.2))
ax.plot(truth,label='true',color='black',lw=2)
for n,c in [('LSTM','#4C72B0'),('Informer','#55A868'),('PatchTST','#DD8452')]: ax.plot(preds[n],label=n,color=c,ls='--')
ax.set_title('未来24h 油温曲线：一条样本的贴合度'); ax.legend()
plt.tight_layout(); plt.savefig(FIGS/'fig4_curve.png',dpi=150,bbox_inches='tight'); plt.show()
print('SUMMARY',[round(res[k],4) for k in ['LSTM','Informer','PatchTST']],[round(float(np.sqrt(res[k])*sd),3) for k in ['LSTM','Informer','PatchTST']])

## 4. 消融：Informer 稀疏关掉，差多少

## 5. 总结与下一步

三路线同台 MSE + OT-RMSE 双口径 + 曲线贴合 + 稀疏消融。下一步 `02_Recommendation_MovieLens`：DeepFM/DIN（MovieLens-100K，需手动下载）。